In [3]:
import pandas as pd
import numpy as np

In [6]:
df = pd.read_excel('ML_Preprocessing_Practice_Dataset.csv')

In [7]:
df

,Education_Level,City,Department,Employment_Type,Age,Monthly_Income,Performance
0,High School,Hyderabad,Marketing,Part-Time,27.0,21250.0,Average
1,Bachelor,Chennai,Sales,Part-Time,24.0,45231.0,Average
2,High School,Chennai,Sales,Contract,36.0,25366.0,Average
3,High School,Bengaluru,IT,Contract,45.0,22289.0,Average
4,High School,Bengaluru,IT,Full-Time,26.0,24658.0,Average
...,...,...,...,...,...,...,...
5525,Bachelor,Mumbai,Marketing,Full-Time,46.0,53613.0,Good
5526,Diploma,Chennai,HR,Contract,33.0,30586.0,Average
5527,Master,Hyderabad,HR,Contract,46.0,79901.0,Good
5528,Master,Bengaluru,Sales,Part-Time,28.0,61100.0,Good


In [8]:
df.head()

,Education_Level,City,Department,Employment_Type,Age,Monthly_Income,Performance
0,High School,Hyderabad,Marketing,Part-Time,27.0,21250.0,Average
1,Bachelor,Chennai,Sales,Part-Time,24.0,45231.0,Average
2,High School,Chennai,Sales,Contract,36.0,25366.0,Average
3,High School,Bengaluru,IT,Contract,45.0,22289.0,Average
4,High School,Bengaluru,IT,Full-Time,26.0,24658.0,Average


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5530 entries, 0 to 5529
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Education_Level  5524 non-null   object 
 1   City             5517 non-null   object 
 2   Department       5519 non-null   object 
 3   Employment_Type  5522 non-null   object 
 4   Age              5515 non-null   float64
 5   Monthly_Income   5523 non-null   float64
 6   Performance      5530 non-null   object 
dtypes: float64(2), object(5)
memory usage: 302.6+ KB


In [12]:
df.isnull().sum()

Education_Level     6
City               13
Department         11
Employment_Type     8
Age                15
Monthly_Income      7
Performance         0
dtype: int64

### ***Handle Missing Values***

> Since this is a categorical column, we'll use the most frequent value (mode).

In [17]:
mode_value = df["Education_Level"].mode()[0]
mode_value

'Bachelor'

In [18]:
df["Education_Level"] = df["Education_Level"].fillna(mode_value)

In [19]:
df["Education_Level"].isnull().sum()

0

###  ***Observe the Column***

> We want to encode only the Education_Level column.

In [9]:
df['Education_Level'].unique()

array(['High School', 'Bachelor', 'Diploma', 'Master', 'PhD', nan],
      dtype=object)

## ***1. Manual Ordinal Encoding ==> Create a Mapping Dictionary***
> Since Education_Level has a natural order, we manually define its ranking.

In [20]:
education_mapping = {
    "High School": 0,
    "Diploma": 1,
    "Bachelor": 2,
    "Master": 3,
    "PhD": 4
}

In [21]:
"""
Why did we choose this order?
Because the education levels naturally progress as:

High School
      ↓
Diploma
      ↓
Bachelor
      ↓
Master
      ↓
PhD
"""

'\nWhy did we choose this order?\nBecause the education levels naturally progress as:\n\nHigh School\n      ↓\nDiploma\n      ↓\nBachelor\n      ↓\nMaster\n      ↓\nPhD\n'

## ***Apply the Mapping***

In [22]:
df["Education_Level"] = df["Education_Level"].map(education_mapping)

In [23]:
df.head()

,Education_Level,City,Department,Employment_Type,Age,Monthly_Income,Performance
0,0,Hyderabad,Marketing,Part-Time,27.0,21250.0,Average
1,2,Chennai,Sales,Part-Time,24.0,45231.0,Average
2,0,Chennai,Sales,Contract,36.0,25366.0,Average
3,0,Bengaluru,IT,Contract,45.0,22289.0,Average
4,0,Bengaluru,IT,Full-Time,26.0,24658.0,Average


### ***Verify the Encoding***

In [24]:
df["Education_Level"].unique()

array([0, 2, 1, 3, 4], dtype=int64)

> here We manually converted the ordinal categories into numerical values using a dictionary and the map() function.

> This approach works well for a single column, but when working with multiple columns or building reusable machine learning pipelines, OrdinalEncoder provides a cleaner and more scalable solution.

# ***2. Ordinal Encoding Using OrdinalEncoder***
> Instead of creating a mapping dictionary manually, Scikit-learn provides the OrdinalEncoder class to perform ordinal encoding automatically.

In [33]:
# reload the dataset
df = pd.read_excel("ML_Preprocessing_Practice_Dataset.csv")

In [34]:
# handle missing values 
mode_value = df["Education_Level"].mode()[0]
df["Education_Level"] = df["Education_Level"].fillna(mode_value)

### ***Step 1 : import the class***

In [35]:
from sklearn.preprocessing import OrdinalEncoder

### ***Step 2: Create the Encoder Object***

> Since OrdinalEncoder does not know the correct order, we must specify the category order manually.

In [36]:
encoder = OrdinalEncoder(categories=[[
    "High School",
    "Diploma",
    "Bachelor",
    "Master",
    "PhD"
]])

> Why do we specify categories?
> Because OrdinalEncoder does not understand the real-world ranking of education levels.
> We explicitly tell it the correct order:

- High School
  -    ↓
- Diploma
  -    ↓
- Bachelor
  -    ↓
- Master
  -    ↓
- PhD

### ***Step 3: Apply the Encoder***

In [37]:
df["Education_Level"] = encoder.fit_transform(
    df[["Education_Level"]]
)

#### ***Notice that we use double square brackets because OrdinalEncoder expects a 2D input (a DataFrame), not a Series.***

## ***Step 4: View the Result***

In [48]:
df['Education_Level'] = df['Education_Level'].astype('int')
df[['Education_Level']]

,Education_Level
0,0
1,2
2,0
3,0
4,0
...,...
5525,2
5526,1
5527,3
5528,3


## ***Step 5: Check the Encoded Values***

In [49]:
df["Education_Level"].unique()

array([0, 2, 1, 3, 4])

# ***Manual vs OrdinalEncoder***
| Manual Method                | `OrdinalEncoder`                                         |
| ---------------------------- | -------------------------------------------------------- |
| Create a dictionary manually | Create an `OrdinalEncoder` object                        |
| Use `map()`                  | Use `fit_transform()`                                    |
| Good for understanding       | Recommended in scikit-learn workflows                    |
| Less suitable for pipelines  | Works seamlessly with `Pipeline` and `ColumnTransformer` |


# ***Important Note***

-  In real machine learning projects, you should fit the encoder only on the training data and then transform both the training and testing data. For simplicity, we've applied it directly to the DataFrame here to focus on understanding how OrdinalEncoder works.